# Notebook 49 — Functional Universality: b as a Functional of Γ_eff(x)

This notebook regenerates Notebook 49 as a visual, publication-style bridge beyond Notebook 48.

Notebook 48 showed that the stretching exponent `b` is better predicted by:

- rate heterogeneity (`CV`)
- plus structured features of the effective-rate process `Γ_eff(x)`

This notebook pushes one step further:

> Can `b` depend on the **shape** of `Γ_eff(x)` even when simple distribution summaries are similar?

We build controlled families of `Γ_eff(x)` curves, generate the corresponding decay laws

`dS/dx = -Γ_eff(x) S`

fit stretched exponentials

`S(x) ≈ exp(-a x^b)`

and test whether `b` tracks the full structure of `Γ_eff(x)`.

## Main goals

1. define several controlled `Γ_eff(x)` families,
2. build the implied decay curves `S(x)`,
3. fit stretched exponentials and extract `b`,
4. compare shape descriptors of `Γ_eff(x)` to the fitted `b`,
5. show visually that functional structure matters beyond scalar summaries.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit


## Grid

In [ ]:
x = np.linspace(1e-3, 1.0, 400)
dx = x[1] - x[0]


## Controlled families of effective-rate processes

In [ ]:
def gamma_flat(x, a=2.0):
    return a * np.ones_like(x)

def gamma_linear_up(x, a=1.6, m=0.9):
    return a + m * x

def gamma_linear_down(x, a=2.4, m=0.9):
    return a - m * x

def gamma_power_up(x, a=1.5, p=0.55):
    return a + 0.9 * np.power(x, p)

def gamma_power_down(x, a=2.4, p=0.55):
    return a - 0.9 * np.power(x, p)

def gamma_curved_convex(x, a=1.4):
    return a + 0.3 * x + 0.9 * x**2

def gamma_curved_concave(x, a=1.5):
    return a + 1.1 * np.sqrt(x)

def gamma_wavy_small(x, a=1.8):
    return a + 0.22 * np.sin(2 * np.pi * x)

def gamma_wavy_large(x, a=1.8):
    return a + 0.45 * np.sin(2 * np.pi * x)

families = {
    "flat": gamma_flat(x),
    "linear_up": gamma_linear_up(x),
    "linear_down": gamma_linear_down(x),
    "power_up": gamma_power_up(x),
    "power_down": gamma_power_down(x),
    "convex": gamma_curved_convex(x),
    "concave": gamma_curved_concave(x),
    "wavy_small": gamma_wavy_small(x),
    "wavy_large": gamma_wavy_large(x),
}


## Build S(x) from Γ_eff(x)

In [ ]:
def build_S_from_gamma(gamma_x, x):
    gamma_x = np.asarray(gamma_x, dtype=float)
    # log S(x) = - ∫ Γ(u) du
    integral = np.cumsum(gamma_x) * (x[1] - x[0])
    S = np.exp(-integral)
    return S


## Stretched-exponential fit

In [ ]:
def stretched(x, a, b):
    return np.exp(-a * np.power(x, b))

def fit_stretched(x, S):
    popt, _ = curve_fit(
        stretched,
        np.clip(x, 1e-12, None),
        np.clip(S, 1e-12, 1.0),
        p0=[1.0, 1.0],
        bounds=([0.0, 0.1], [100.0, 5.0]),
        maxfev=50000,
    )
    a, b = [float(v) for v in popt]
    pred = stretched(x, a, b)
    mse = float(np.mean((S - pred) ** 2))
    return a, b, pred, mse


## Shape descriptors for Γ_eff(x)

In [ ]:
def gamma_features(gamma_x, x):
    gamma_x = np.asarray(gamma_x, dtype=float)
    d1 = np.gradient(gamma_x, x)
    d2 = np.gradient(d1, x)

    mean = float(np.mean(gamma_x))
    std = float(np.std(gamma_x))
    cv = float(std / mean) if mean > 0 else np.nan
    slope_mean = float(np.mean(d1))
    slope_abs = float(np.mean(np.abs(d1)))
    curvature_abs = float(np.mean(np.abs(d2)))
    dynamic_range = float(np.max(gamma_x) - np.min(gamma_x))
    roughness = float(np.std(np.diff(gamma_x)))
    area = float(np.trapz(gamma_x, x))

    return {
        "mean": mean,
        "std": std,
        "cv": cv,
        "slope_mean": slope_mean,
        "slope_abs": slope_abs,
        "curvature_abs": curvature_abs,
        "dynamic_range": dynamic_range,
        "roughness": roughness,
        "area": area,
    }


## Evaluate all families

In [ ]:
results = []

for name, gamma_x in families.items():
    S = build_S_from_gamma(gamma_x, x)
    a_fit, b_fit, S_fit, mse = fit_stretched(x, S)
    feats = gamma_features(gamma_x, x)

    results.append({
        "name": name,
        "gamma": gamma_x,
        "S": S,
        "a_fit": a_fit,
        "b_fit": b_fit,
        "S_fit": S_fit,
        "fit_mse": mse,
        **feats,
    })

for row in results:
    print(
        row["name"],
        "b=", f'{row["b_fit"]:.4f}',
        "cv=", f'{row["cv"]:.4f}',
        "range=", f'{row["dynamic_range"]:.4f}',
        "curv=", f'{row["curvature_abs"]:.4f}',
        "mse=", f'{row["fit_mse"]:.3e}'
    )


## Plot 1 — Effective-rate processes Γ_eff(x)

In [ ]:
plt.figure(figsize=(8.4, 5.3))
for row in results:
    plt.plot(x, row["gamma"], label=row["name"])
plt.xlabel("x")
plt.ylabel("Γ_eff(x)")
plt.title("Families of effective-rate processes")
plt.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()


## Plot 2 — Resulting decay curves S(x)

In [ ]:
plt.figure(figsize=(8.4, 5.3))
for row in results:
    plt.plot(x, row["S"], label=row["name"])
plt.xlabel("x")
plt.ylabel("S(x)")
plt.title("Decay curves generated by Γ_eff(x)")
plt.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()


## Plot 3 — Stretched-exponential fits

In [ ]:
plt.figure(figsize=(8.6, 5.5))
for row in results:
    plt.plot(x, row["S"], linewidth=2.0, alpha=0.9, label=f'{row["name"]} data')
    plt.plot(x, row["S_fit"], linestyle="--", linewidth=1.5, label=f'{row["name"]} fit (b={row["b_fit"]:.2f})')
plt.xlabel("x")
plt.ylabel("S(x)")
plt.title("Stretched-exponential fits to functional-rate decays")
plt.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()


## Plot 4 — Same CV, different shape → different b

In [ ]:
# Find pairs with relatively similar CV but different structural descriptors
cv_vals = np.array([row["cv"] for row in results], dtype=float)
b_vals = np.array([row["b_fit"] for row in results], dtype=float)
curv_vals = np.array([row["curvature_abs"] for row in results], dtype=float)

plt.figure(figsize=(7.4, 5.0))
plt.scatter(cv_vals, b_vals, s=90)
for row in results:
    plt.annotate(row["name"], (row["cv"], row["b_fit"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
plt.xlabel("CV of Γ_eff(x)")
plt.ylabel("fitted b")
plt.title("CV alone does not fully determine b")
plt.tight_layout()
plt.show()


## Plot 5 — b vs slope / curvature / range

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.6))

features_to_plot = [
    ("slope_abs", "mean |dΓ/dx|"),
    ("curvature_abs", "mean |d²Γ/dx²|"),
    ("dynamic_range", "max Γ - min Γ"),
]

for ax, (feature_key, feature_label) in zip(axes, features_to_plot):
    vals = np.array([row[feature_key] for row in results], dtype=float)
    ax.scatter(vals, b_vals, s=80)
    for row in results:
        ax.annotate(row["name"], (row[feature_key], row["b_fit"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
    ax.set_xlabel(feature_label)
    ax.set_ylabel("b")
    ax.set_title(f"b vs {feature_label}")

plt.tight_layout()
plt.show()


## Plot 6 — Shape descriptors compared

In [ ]:
names = [row["name"] for row in results]
slope_abs = np.array([row["slope_abs"] for row in results], dtype=float)
curv_abs = np.array([row["curvature_abs"] for row in results], dtype=float)
dyn_range = np.array([row["dynamic_range"] for row in results], dtype=float)

# z-score for visual comparability
def zscore(v):
    v = np.asarray(v, dtype=float)
    s = np.std(v)
    return (v - np.mean(v)) / s if s > 0 else v - np.mean(v)

plt.figure(figsize=(8.5, 5.0))
plt.plot(names, zscore(slope_abs), marker="o", label="slope_abs")
plt.plot(names, zscore(curv_abs), marker="o", label="curvature_abs")
plt.plot(names, zscore(dyn_range), marker="o", label="dynamic_range")
plt.plot(names, zscore(b_vals), marker="o", label="b")
plt.xticks(rotation=30)
plt.ylabel("z-scored value")
plt.title("Comparing shape descriptors to fitted b")
plt.legend()
plt.tight_layout()
plt.show()


## Plot 7 — Functional universality map

In [ ]:
# Build a 2D map using curvature and range as illustrative shape axes
range_vals = np.array([row["dynamic_range"] for row in results], dtype=float)

plt.figure(figsize=(7.5, 5.3))
sc = plt.scatter(curv_vals, range_vals, c=b_vals, s=130)
for row in results:
    plt.annotate(row["name"], (row["curvature_abs"], row["dynamic_range"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
plt.colorbar(sc, label="fitted b")
plt.xlabel("mean |d²Γ/dx²|")
plt.ylabel("dynamic range of Γ_eff")
plt.title("Functional universality map")
plt.tight_layout()
plt.show()


## Compact summary

In [ ]:
lines = []
lines.append("Functional universality: b as a functional of Γ_eff(x)")
lines.append("")
for row in results:
    lines.append(
        f'- {row["name"]}: b={row["b_fit"]:.6f}, CV={row["cv"]:.6f}, '
        f'slope_abs={row["slope_abs"]:.6f}, curvature_abs={row["curvature_abs"]:.6f}, '
        f'dynamic_range={row["dynamic_range"]:.6f}, fit_MSE={row["fit_mse"]:.6e}'
    )
lines.append("")
lines.append("Interpretation:")
lines.append("- Different shapes of Γ_eff(x) generate different decay laws S(x).")
lines.append("- Those decay laws remain well fit by stretched exponentials.")
lines.append("- However, b is not fixed by CV alone.")
lines.append("- Shape descriptors such as slope, curvature, and dynamic range also affect b.")
lines.append("- This supports the functional upgrade: b = Functional[Γ_eff(x)]")

print("\n".join(lines))


## Final conclusion

This notebook upgrades the universality claim from a feature-based model to a functional one.

The key result is:

- scalar summaries such as `CV` help,
- but the **shape of the effective-rate process** `Γ_eff(x)` matters directly,
- therefore the stretching exponent is better understood as a **functional of Γ_eff(x)**.

This is the natural continuation after Notebook 48 because it shows visually and quantitatively
that dynamical universality is governed by the structure of the full rate process, not only by low-order summaries.
